# Laser phase-error output: ADC 224/3 → DAC 229/1

Keep `laser_pll.bit`, its matching `laser_pll.hwh`, `laser_pll.py` and this notebook in the same directory on the RFSoC/PYNQ board. Use the existing board-specific reference-clock initialization procedure at its usual point in your overlay-loading workflow. This design expects ADC tile 224 at 1.96608 GS/s and a 245.76 MHz fabric clock; the notebook does not change sample clocks.

The output is an error voltage for an **external analog controller**. The DAC/board connection must carry DC. Configuration starts muted. Run the enable cell only when actuator polarity and gain are set appropriately. Software status is a live diagnostic, not a lock detector. Avoid **Run All**, which would immediately execute acquisition, capture-disable, retuning and mute examples in succession.


## 1. Load the matching overlay
Run from the directory containing the four files. Loading reprograms the FPGA. Do not run the old ADC mixer/decimator test configuration afterward.


In [ ]:
from pathlib import Path
import sys
from pynq import Overlay

overlay_dir = Path.cwd()
sys.path.insert(0, str(overlay_dir))
import laser_pll

# Use your existing board clock initialization at its normal point in this workflow.
overlay = Overlay(str(overlay_dir / "laser_pll.bit"))


## 2. Configure while muted
The reference is a fabric DDS. `configure` also resets the DAC NCO to zero frequency/phase so it passes the baseband error. Gain values are shifts: 0 means ×1, 4 means ×16. The amplitude threshold is in post-mixer ADC counts (approximately half the input peak amplitude near zero residual frequency).


In [ ]:
reference_frequency_hz = 800e6
actual_frequency_hz = laser_pll.configure(
    overlay,
    reference_frequency_hz=reference_frequency_hz,
    phase_offset_rad=0.0,
    phase_gain_shift=0,
    capture_gain_shift=6,
    minimum_amplitude=64,
    invert=False,  # Set for the negative-feedback polarity of your analog actuator.
)
print(f"Programmed reference: {actual_frequency_hz:.6f} Hz")
print(laser_pll.status(overlay))  # enabled=False and valid=False are expected while muted.


## 3. Enable acquisition output
For an input above the reference, the default frequency error is negative. Shift 6 weights the phase increment by 64. At ±100 MHz the output saturates with the corresponding acquisition polarity; actual pull-in depends on the analog actuator/controller.


In [ ]:
laser_pll.enable_capture(overlay, True)
laser_pll.enable(overlay, True)
print(laser_pll.status(overlay))


## 4. Read diagnostics
`valid` means a populated pipeline with sufficient signal amplitude. `saturated` means the requested DAC error exceeded its range. `dac_stall_seen` flags a break in normal continuous DAC acceptance. Phase and frequency are read separately, so they are not a simultaneous snapshot.


In [ ]:
diagnostics = laser_pll.status(overlay)
print(f"Phase error: {diagnostics['phase_error_rad']:.6g} rad")
print(f"Frequency error: {diagnostics['frequency_error_hz']:.6g} Hz")
print(diagnostics)


## 5. After lock has been established externally
Disable the acquisition term for phase-only error. This changes detector gain/phase as well as noise; tune the locked analog loop for this mode. There is no automatic lock decision in this cell.


In [ ]:
laser_pll.enable_capture(overlay, False)


## 6. Adjust gains
This example sets phase gain ×16, giving about 24 µrad per nominal physical DAC step and an unsaturated phase-only range of approximately ±π/16. Both gain fields change together; enable, capture and polarity are preserved. **Set the desired gain before tuning/enabling the analog loops:** changing shift 0 to 4 is a 24.1 dB detector-gain step. The function does not hold or retune the analog controllers.


In [ ]:
laser_pll.set_gains(overlay, phase_gain_shift=4, capture_gain_shift=6)


## 7. Optional phase-continuous frequency change
Ordinary writes preserve accumulated DDS phase. Python/GPIO timing is not deterministic, and the present FIR/capture specifications apply around the 800 MHz reference. Phase-coherent tracking of a separately changing source requires a shared hardware trajectory/epoch.


In [ ]:
actual_frequency_hz = laser_pll.set_reference_frequency(overlay, 800.001e6)
print(f"Programmed reference: {actual_frequency_hz:.6f} Hz")


## 8. Mute
This zeros the error output and clears detector history while the reference DDS continues advancing. Call `configure` again while muted to change the phase setpoint or polarity cleanly; it also restarts the reference phase.


In [ ]:
laser_pll.enable(overlay, False)
print(laser_pll.status(overlay))


## 9. Independent bench test with a fixed 800 MHz generator
The custom DC interface was already verified with the previous constant-output setup. Keep the laser controller disconnected and observe this interface on a DC-coupled oscilloscope. Use a generator amplitude above the detector threshold and below ADC clipping. Run each following cell separately, pausing to inspect the scope.

At reference 799.9 MHz and capture off, expect a downward 10 µs sawtooth (upward wrap), relative to the interface polarity. At reference 800.1 MHz the ramp reverses. Exactly matched frequencies give an arbitrary phase-dependent DC level; clock mismatch makes it drift. These are open-loop tests, not lock acquisition tests.


In [ ]:
laser_pll.configure(overlay, reference_frequency_hz=799.9e6,
                    phase_gain_shift=0, capture_gain_shift=6,
                    minimum_amplitude=64, invert=False)
laser_pll.enable_capture(overlay, False)
laser_pll.enable(overlay, True)
print(laser_pll.status(overlay))


In [ ]:
laser_pll.set_reference_frequency(overlay, 800.1e6)
print(laser_pll.status(overlay))


### Wide-range capture polarity
With the generator still at 800 MHz and phase gain 1, enable capture. At a 700 MHz reference expect negative full scale and a frequency diagnostic near −100 MHz. At 900 MHz expect positive full scale and a diagnostic near +100 MHz. Interface inversion reverses measured voltage polarity. The phase term keeps wrapping internally even when the DAC is saturated.


In [ ]:
laser_pll.enable_capture(overlay, True)
laser_pll.set_reference_frequency(overlay, 700e6)
print(laser_pll.status(overlay))


In [ ]:
laser_pll.set_reference_frequency(overlay, 900e6)
print(laser_pll.status(overlay))


### End the bench test
Mute the output before reconnecting or changing the experimental setup.


In [ ]:
laser_pll.enable(overlay, False)
print(laser_pll.status(overlay))


## Piezo / intracavity-EOM handoff

The piezo controller **stays on**. The EOM is inside the laser cavity. Both controllers have integrator limits; reset requires off/on. There is one shared DAC error signal, and these cells do not operate the analog controllers.

1. Keep the EOM controller off/reset with its tracking filters and low initial gain selected. Let the active piezo acquire with FPGA capture enabled.
2. Observe frequency **excursion**, piezo command and saturation over several settling times. A zero crossing or zero signed mean is not acquisition. The entire excursion must fit the fast loop's measured pull-in capability; piezo-only phase lock is helpful but not mandatory.
3. As settling permits, reduce capture weight to a small value while keeping phase gain fixed. `capture_gain_shift=0` means weight 1, not disabled. **Do not reset the piezo controller** or switch a filter that discards its stored tuning command.
4. Engage the EOM controller at low gain with filters selected for the final phase-only loop. After phase acquisition, disable the remaining capture term and verify stability before increasing fast gain.
5. For reacquisition, turn off/reset the EOM before restoring a large shared capture error. Preserve the piezo tuning command where possible.

At zero frequency error and constant phase, capture contributes zero. But the default weight 64 adds +5.70 dB and +58.04 degrees to the detector response at 1 MHz for phase fluctuations, even at zero mean frequency offset. Turning it off can substantially alter the fast loop. Weight 1 adds only +0.0057 dB and +1.46 degrees at 1 MHz, making the final capture-off transition much smaller. These are detector transfer-function changes, not guarantees of closed-loop phase margin.

At weight 64 the error fully saturates beyond about ±3.84 MHz. An integrating piezo loop can overshoot repeatedly even with limits. Check sign and reduce excessive integral/capture gain if it does not settle. A 1 MHz fast bandwidth does not imply a 1 MHz pull-in range. See `docs/laser_pll.md` for the detailed procedure.


## Read-only frequency-excursion diagnostic
Run this while the piezo acquisition is active, after the manual setup above. It only reads registers. It plots `f_input - f_reference`, the negative of the helper's frequency-error sign. Invalid samples are omitted.

Python/MMIO sampling is irregular and much slower than the detector. Check the measured polling intervals against the oscillation period; this trace can miss faster excursions or cycle slips and is **not** an automatic handoff criterion. Use a fast hardware capture/scope measurement when the motion is not resolved. No signed-mean lock decision is made.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

samples = []
start_s = time.monotonic()
for _ in range(500):
    state = laser_pll.status(overlay)
    samples.append((time.monotonic() - start_s,
                    -state["frequency_error_hz"], state["valid"]))

time_s, frequency_offset_hz, valid = np.asarray(samples).T
valid = valid.astype(bool)
interval_s = np.diff(time_s)
print(f"Median polling interval: {np.median(interval_s)*1e6:.1f} us; "
      f"largest: {interval_s.max()*1e6:.1f} us")
if valid.any():
    observed = frequency_offset_hz[valid]
    print(f"Observed frequency offset: {observed.min()/1e6:+.6f} to "
          f"{observed.max()/1e6:+.6f} MHz (may miss faster excursions)")
else:
    print("No valid samples: check output enable, input amplitude, clock and reset.")
plt.figure(figsize=(8, 3))
plt.plot(time_s*1e3, np.where(valid, frequency_offset_hz/1e6, np.nan), ".-")
plt.xlabel("Time (ms)")
plt.ylabel("Input − reference (MHz)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
